In [8]:
from pathlib import Path
import time

import pandas as pd
import joblib

import optuna
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

SEED = 42

In [9]:
PROJECT_ROOT = Path(r"C:\Case Study Trees\HomeCreditStudy")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "native"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "native"
RESULT_DIR = PROJECT_ROOT / "results" / "native"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


y_train = pd.read_csv(
    PROCESSED_DIR / "engineered" / "y_train.csv"
)["TARGET"]

y_valid = pd.read_csv(
    PROCESSED_DIR / "engineered" / "y_valid.csv"
)["TARGET"]

print("Training labels:", len(y_train))
print("Validation labels:", len(y_valid))

STUDY_DIR = PROJECT_ROOT / "studies"
STUDY_DIR.mkdir(parents=True, exist_ok=True)
database_path = (STUDY_DIR / "home_credit_native_optuna.db").resolve()

for split in ["train", "valid"]:
    reference_y = pd.read_csv(PROCESSED_DIR / "engineered" / f"y_{split}.csv")
    filtered_y = pd.read_csv(PROCESSED_DIR / "engineered_filtered" / f"y_{split}.csv")
    pd.testing.assert_frame_equal(reference_y, filtered_y)
    reference_ids = pd.read_csv(PROCESSED_DIR / "engineered" / f"{split}_ids.csv")
    filtered_ids = pd.read_csv(PROCESSED_DIR / "engineered_filtered" / f"{split}_ids.csv")
    pd.testing.assert_frame_equal(reference_ids, filtered_ids)
    assert len(reference_ids) == len(reference_y)
    assert reference_ids["SK_ID_CURR"].is_unique
print("Both feature versions have matching labels and IDs.")


Training labels: 215256
Validation labels: 46127
Both feature versions have matching labels and IDs.


## XGBoost
### Load engineered data and display categorical features


In [10]:
from xgboost import XGBClassifier

feature_folder = 'engineered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"xgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)


Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 190)
Validation: (46127, 190)


### Train the engineered baseline

In [11]:
xgb_engineered = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
        enable_categorical=True,
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

xgb_engineered.fit(X_train, y_train)

xgb_engineered_fit_seconds = time.perf_counter() - start_time

print(
    "Training time:",
    round(xgb_engineered_fit_seconds, 2),
    "seconds"
)

Training time: 16.09 seconds


### Evaluate on validation and save the baseline

In [12]:
start_time = time.perf_counter()

valid_probability = xgb_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_engineered_results = {
    "Model": "XGBoost",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([xgb_engineered_results]).round(4))

joblib.dump(
    xgb_engineered,
    ARTIFACT_DIR / "xgb_engineered_baseline.joblib"
)

pd.DataFrame([xgb_engineered_results]).to_csv(
    RESULT_DIR / "xgb_engineered_baseline_validation.csv",
    index=False
)
# Portable save retains categorical model information.
xgb_engineered.save_model(ARTIFACT_DIR / "xgb_engineered_baseline.ubj")


,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,XGBoost,Engineered,0.7592,0.2475,0.2457,16.0891,0.2932


### Load filtered data, display categorical features and train


In [13]:
feature_folder = 'engineered_filtered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)


input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"xgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

xgb_filtered = XGBClassifier(**xgb_engineered.get_params())

start_time = time.perf_counter()

xgb_filtered.fit(X_train, y_train)

xgb_filtered_fit_seconds = time.perf_counter() - start_time

print("Training features:", X_train.shape)
print(
    "Training time:",
    round(xgb_filtered_fit_seconds, 2),
    "seconds"
)

Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 134)
Validation: (46127, 134)
Training features: (215256, 134)
Training time: 12.84 seconds


### Compare both baselines and save the filtered model

In [14]:
start_time = time.perf_counter()

valid_probability = xgb_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_filtered_results = {
    "Model": "XGBoost",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

xgb_comparison = pd.DataFrame([
    xgb_engineered_results,
    xgb_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(xgb_comparison.round(4))

joblib.dump(
    xgb_filtered,
    ARTIFACT_DIR / "xgb_filtered_baseline.joblib"
)

xgb_comparison.to_csv(
    RESULT_DIR / "xgb_baseline_comparison.csv",
    index=False
)
xgb_filtered.save_model(ARTIFACT_DIR / "xgb_filtered_baseline.ubj")


,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
1,XGBoost,Engineered + filtered,0.7598,0.2483,0.2455,12.8365,0.2795
0,XGBoost,Engineered,0.7592,0.2475,0.2457,16.0891,0.2932


### Select the better feature version automatically


In [15]:
if xgb_filtered_results["ROC_AUC"] > xgb_engineered_results["ROC_AUC"]:
    xgb_feature_folder = "engineered_filtered"
    xgb_feature_label = "Engineered + filtered"
    xgb_selected_baseline = xgb_filtered_results
else:
    xgb_feature_folder = "engineered"
    xgb_feature_label = "Engineered"
    xgb_selected_baseline = xgb_engineered_results

print("Selected:", xgb_feature_label)
print("Baseline validation ROC-AUC:", xgb_selected_baseline["ROC_AUC"])

feature_folder = xgb_feature_folder

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"xgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

pd.DataFrame([{
    "Model": "xgb", "Feature_Folder": xgb_feature_folder,
    "Input_Info_File": f"xgb_{xgb_feature_folder}_input_info.joblib",
    "Baseline_Model_File": "xgb_filtered_baseline.joblib" if xgb_feature_folder == "engineered_filtered" else "xgb_engineered_baseline.joblib",
    "Tuned_Model_File": f"xgb_{xgb_feature_folder}_tuned.joblib"
}]).to_csv(RESULT_DIR / "xgb_selected_configuration.csv", index=False)


Selected: Engineered + filtered
Baseline validation ROC-AUC: 0.7598387187761805
Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 134)
Validation: (46127, 134)


### Optuna objective and 30-minute study


In [16]:
def xgb_objective(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        max_depth=trial.suggest_int("max_depth", 3, 9),
        min_child_weight=trial.suggest_float(
            "min_child_weight", 1, 30, log=True
        ),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        random_state=SEED,
        n_jobs=4
    )

    start = time.perf_counter()

    model.fit(X_train, y_train)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)


xgb_study = optuna.create_study(
    study_name=f"xgb_native_{xgb_feature_folder}_matched_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

xgb_study.optimize(
    xgb_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

xgb_tuning_seconds = time.perf_counter() - tuning_start

xgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "xgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", xgb_study.best_value)
print("Best parameters:", xgb_study.best_params)
print("Tuning seconds:", round(xgb_tuning_seconds, 2))

[I 2026-10-16 03:29:23,137] A new study created in RDB with name: xgb_native_engineered_filtered_matched_v1
[I 2026-10-16 03:29:51,808] Trial 0 finished with value: 0.7358267721821052 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.6611007077713635, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.7358267721821052.
[I 2026-10-16 03:30:19,610] Trial 1 finished with value: 0.7627107979789747 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'max_depth': 3, 'min_child_weight': 27.08160864249967, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.7627107979789747.
[I 2026-10-16 03:30:38,349] Trial 2 finished with value: 0.7616111313942711 and parameters: {

Best validation ROC-AUC: 0.7650229906163794
Best parameters: {'n_estimators': 800, 'learning_rate': 0.021830147868626797, 'max_depth': 5, 'min_child_weight': 28.94327996313511, 'subsample': 0.8131679446731267, 'colsample_bytree': 0.6317964966622504, 'reg_alpha': 0.00016968369429841315, 'reg_lambda': 6.9920290933264235}
Tuning seconds: 1801.34


### Train the selected parameters, evaluate and save.

In [17]:
xgb_tuned = XGBClassifier(
    **xgb_study.best_params,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
        enable_categorical=True,
    random_state=SEED,
    n_jobs=4
)

start_time = time.perf_counter()

xgb_tuned.fit(X_train, y_train)

xgb_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = xgb_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

xgb_tuned_results = {
    "Model": "XGBoost",
    "Version": "Tuned",
    "Feature_Set": xgb_feature_label,
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": xgb_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": xgb_tuning_seconds
}

display(pd.DataFrame([xgb_tuned_results]).round(4))

joblib.dump(
    xgb_tuned,
    ARTIFACT_DIR / f"xgb_{xgb_feature_folder}_tuned.joblib"
)

pd.DataFrame([xgb_tuned_results]).to_csv(
    RESULT_DIR / "xgb_tuned_validation.csv",
    index=False
)

pd.Series(xgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "xgb_best_parameters.csv",
    index_label="Parameter"
)
xgb_tuned.save_model(ARTIFACT_DIR / f"xgb_{xgb_feature_folder}_tuned.ubj")


,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,XGBoost,Tuned,Engineered + filtered,0.765,0.2566,0.2439,30.7478,0.4556,1801.3413


## LightGBM
### Load engineered data and display categorical features


In [18]:
from lightgbm import LGBMClassifier

feature_folder = 'engineered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

safe_names = [f"feature_{i}" for i in range(X_train.shape[1])]
name_mapping = dict(zip(original_columns, safe_names))
feature_mapping = pd.DataFrame({
    "Original_Name": original_columns, "LightGBM_Name": safe_names
})
feature_mapping.to_csv(
    ARTIFACT_DIR / f"lgb_{feature_folder}_feature_mapping.csv", index=False
)
X_train.columns = safe_names
X_valid.columns = safe_names
cat_columns = [name_mapping[column] for column in cat_columns]
input_info["model_columns"] = safe_names
input_info["categorical_model_columns"] = cat_columns

joblib.dump(input_info, ARTIFACT_DIR / f"lgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)


Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 190)
Validation: (46127, 190)


### Train the engineered baseline

In [19]:
lgb_engineered = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1
)

start_time = time.perf_counter()

lgb_engineered.fit(X_train, y_train, categorical_feature=cat_columns)

lgb_engineered_fit_seconds = time.perf_counter() - start_time

print(
    "Training time:",
    round(lgb_engineered_fit_seconds, 2),
    "seconds"
)

Training time: 9.83 seconds


### Evaluate on validation and save the baseline

In [20]:
start_time = time.perf_counter()

valid_probability = lgb_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_engineered_results = {
    "Model": "LightGBM",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([lgb_engineered_results]).round(4))

joblib.dump(
    lgb_engineered,
    ARTIFACT_DIR / "lgb_engineered_baseline.joblib"
)

pd.DataFrame([lgb_engineered_results]).to_csv(
    RESULT_DIR / "lgb_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,LightGBM,Engineered,0.76,0.2481,0.2456,9.8255,0.5505


### Load filtered data, display categorical features and train


In [21]:
feature_folder = 'engineered_filtered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

# Same LightGBM safe-name fix as in the original notebook.
safe_names = [f"feature_{i}" for i in range(X_train.shape[1])]
name_mapping = dict(zip(original_columns, safe_names))
feature_mapping = pd.DataFrame({
    "Original_Name": original_columns, "LightGBM_Name": safe_names
})
feature_mapping.to_csv(
    ARTIFACT_DIR / f"lgb_{feature_folder}_feature_mapping.csv", index=False
)
X_train.columns = safe_names
X_valid.columns = safe_names
cat_columns = [name_mapping[column] for column in cat_columns]
input_info["model_columns"] = safe_names
input_info["categorical_model_columns"] = cat_columns

joblib.dump(input_info, ARTIFACT_DIR / f"lgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

lgb_filtered = LGBMClassifier(**lgb_engineered.get_params())

start_time = time.perf_counter()

lgb_filtered.fit(X_train, y_train, categorical_feature=cat_columns)

lgb_filtered_fit_seconds = time.perf_counter() - start_time

print("Training features:", X_train.shape)
print(
    "Training time:",
    round(lgb_filtered_fit_seconds, 2),
    "seconds"
)

Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 134)
Validation: (46127, 134)
Training features: (215256, 134)
Training time: 6.6 seconds


### Compare both baselines and save the filtered model

In [22]:
start_time = time.perf_counter()

valid_probability = lgb_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_filtered_results = {
    "Model": "LightGBM",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

lgb_comparison = pd.DataFrame([
    lgb_engineered_results,
    lgb_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(lgb_comparison.round(4))

joblib.dump(
    lgb_filtered,
    ARTIFACT_DIR / "lgb_filtered_baseline.joblib"
)

lgb_comparison.to_csv(
    RESULT_DIR / "lgb_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,LightGBM,Engineered,0.7600,0.2481,0.2456,9.8255,0.5505
1,LightGBM,Engineered + filtered,0.7595,0.2494,0.2456,6.5990,0.5119


### Select the better feature version automatically
Use unrounded validation ROC-AUC. Exact ties select engineered. Reload the winner before Optuna; do not assume filtered will win.

In [23]:
if lgb_filtered_results["ROC_AUC"] > lgb_engineered_results["ROC_AUC"]:
    lgb_feature_folder = "engineered_filtered"
    lgb_feature_label = "Engineered + filtered"
    lgb_selected_baseline = lgb_filtered_results
else:
    lgb_feature_folder = "engineered"
    lgb_feature_label = "Engineered"
    lgb_selected_baseline = lgb_engineered_results

print("Selected:", lgb_feature_label)
print("Baseline validation ROC-AUC:", lgb_selected_baseline["ROC_AUC"])

feature_folder = lgb_feature_folder

# Find original categorical columns before assigning any model types.
original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = pd.Categorical(X_train[column], categories=levels)
    X_valid[column] = pd.Categorical(X_valid[column], categories=levels)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

# Same LightGBM safe-name fix as in the original notebook.
safe_names = [f"feature_{i}" for i in range(X_train.shape[1])]
name_mapping = dict(zip(original_columns, safe_names))
feature_mapping = pd.DataFrame({
    "Original_Name": original_columns, "LightGBM_Name": safe_names
})
feature_mapping.to_csv(
    ARTIFACT_DIR / f"lgb_{feature_folder}_feature_mapping.csv", index=False
)
X_train.columns = safe_names
X_valid.columns = safe_names
cat_columns = [name_mapping[column] for column in cat_columns]
input_info["model_columns"] = safe_names
input_info["categorical_model_columns"] = cat_columns

joblib.dump(input_info, ARTIFACT_DIR / f"lgb_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

pd.DataFrame([{
    "Model": "lgb", "Feature_Folder": lgb_feature_folder,
    "Input_Info_File": f"lgb_{lgb_feature_folder}_input_info.joblib",
    "Baseline_Model_File": "lgb_filtered_baseline.joblib" if lgb_feature_folder == "engineered_filtered" else "lgb_engineered_baseline.joblib",
    "Tuned_Model_File": f"lgb_{lgb_feature_folder}_tuned.joblib"
}]).to_csv(RESULT_DIR / "lgb_selected_configuration.csv", index=False)


Selected: Engineered
Baseline validation ROC-AUC: 0.7599586804462007
Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 190)
Validation: (46127, 190)


### Optuna objective and 30-minute study


In [24]:
def lgb_objective(trial):
    model = LGBMClassifier(
        n_estimators=trial.suggest_int(
            "n_estimators", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        max_depth=-1,
        min_child_samples=trial.suggest_int(
            "min_child_samples", 20, 150
        ),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        reg_alpha=trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        reg_lambda=trial.suggest_float(
            "reg_lambda", 0.1, 20, log=True
        ),
        objective="binary",
        random_state=SEED,
        n_jobs=4,
        verbosity=-1
    )

    start = time.perf_counter()

    model.fit(X_train, y_train, categorical_feature=cat_columns)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)


lgb_study = optuna.create_study(
    study_name=f"lgb_native_{lgb_feature_folder}_matched_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

lgb_study.optimize(
    lgb_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

lgb_tuning_seconds = time.perf_counter() - tuning_start

lgb_study.trials_dataframe().to_csv(
    RESULT_DIR / "lgb_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", lgb_study.best_value)
print("Best parameters:", lgb_study.best_params)
print("Tuning seconds:", round(lgb_tuning_seconds, 2))

[I 2026-10-16 04:00:32,398] A new study created in RDB with name: lgb_native_engineered_matched_v1
[I 2026-10-16 04:00:55,553] Trial 0 finished with value: 0.7332404877418717 and parameters: {'n_estimators': 500, 'learning_rate': 0.13125830316209655, 'num_leaves': 97, 'min_child_samples': 98, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 9.842315738502599}. Best is trial 0 with value: 0.7332404877418717.
[I 2026-10-16 04:01:11,519] Trial 1 finished with value: 0.7599567806150755 and parameters: {'n_estimators': 700, 'learning_rate': 0.06803900745073706, 'num_leaves': 17, 'min_child_samples': 147, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 0.0008111941985431928, 'reg_lambda': 0.26425260575499165}. Best is trial 1 with value: 0.7599567806150755.
[I 2026-10-16 04:01:27,433] Trial 2 finished with value: 0.7603103138564082 and parameters: {'n_estimators': 400, 'learning_ra

Best validation ROC-AUC: 0.7628080851645153
Best parameters: {'n_estimators': 600, 'learning_rate': 0.017934053484727538, 'num_leaves': 53, 'min_child_samples': 108, 'subsample': 0.7389742031668991, 'colsample_bytree': 0.892526084888609, 'reg_alpha': 0.255768262778987, 'reg_lambda': 13.85464198133805}
Tuning seconds: 1817.48


### Train the selected parameters, evaluate and save


In [25]:
lgb_tuned = LGBMClassifier(
    **lgb_study.best_params,
    max_depth=-1,
    subsample_freq=1,
    objective="binary",
    random_state=SEED,
    n_jobs=4,
    verbosity=-1
)

start_time = time.perf_counter()

lgb_tuned.fit(X_train, y_train, categorical_feature=cat_columns)

lgb_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = lgb_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

lgb_tuned_results = {
    "Model": "LightGBM",
    "Version": "Tuned",
    "Feature_Set": lgb_feature_label,
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": lgb_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": lgb_tuning_seconds
}

display(pd.DataFrame([lgb_tuned_results]).round(4))

joblib.dump(
    lgb_tuned,
    ARTIFACT_DIR / f"lgb_{lgb_feature_folder}_tuned.joblib"
)

pd.DataFrame([lgb_tuned_results]).to_csv(
    RESULT_DIR / "lgb_tuned_validation.csv",
    index=False
)

pd.Series(lgb_study.best_params, name="Value").to_csv(
    RESULT_DIR / "lgb_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,LightGBM,Tuned,Engineered,0.7628,0.2514,0.2448,23.9386,1.924,1817.4797


## CatBoost
### Load engineered data and display categorical features


In [26]:
from catboost import CatBoostClassifier

feature_folder = 'engineered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

  
    X_train[column] = X_train[column].astype(str)
    X_valid[column] = X_valid[column].astype(str)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"cat_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)


Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 190)
Validation: (46127, 190)


### Train the engineered baseline

In [27]:
cat_baseline_params = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "thread_count": 4,
    "verbose": False,
    "allow_writing_files": False
}

cat_engineered = CatBoostClassifier(**cat_baseline_params)

start_time = time.perf_counter()

cat_engineered.fit(X_train, y_train, cat_features=cat_columns)

cat_engineered_fit_seconds = time.perf_counter() - start_time

print("Training seconds:", round(cat_engineered_fit_seconds, 2))

Training seconds: 153.22


### Evaluate on validation and save the baseline

In [28]:
start_time = time.perf_counter()

valid_probability = cat_engineered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_engineered_results = {
    "Model": "CatBoost",
    "Feature_Set": "Engineered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_engineered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

display(pd.DataFrame([cat_engineered_results]).round(4))

joblib.dump(
    cat_engineered,
    ARTIFACT_DIR / "cat_engineered_baseline.joblib"
)

pd.DataFrame([cat_engineered_results]).to_csv(
    RESULT_DIR / "cat_engineered_baseline_validation.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,CatBoost,Engineered,0.7631,0.2546,0.2445,153.2157,0.4777


### Load filtered data, display categorical features and train


In [29]:
feature_folder = 'engineered_filtered'

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = X_train[column].astype(str)
    X_valid[column] = X_valid[column].astype(str)
    
input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"cat_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

cat_filtered = CatBoostClassifier(**cat_baseline_params)

start_time = time.perf_counter()

cat_filtered.fit(X_train, y_train, cat_features=cat_columns)

cat_filtered_fit_seconds = time.perf_counter() - start_time

print("Training seconds:", round(cat_filtered_fit_seconds, 2))

Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 134)
Validation: (46127, 134)
Training seconds: 147.19


### Compare both baselines and save the filtered model

In [30]:
start_time = time.perf_counter()

valid_probability = cat_filtered.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_filtered_results = {
    "Model": "CatBoost",
    "Feature_Set": "Engineered + filtered",
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_filtered_fit_seconds,
    "Prediction_Seconds": prediction_seconds
}

cat_comparison = pd.DataFrame([
    cat_engineered_results,
    cat_filtered_results
]).sort_values("ROC_AUC", ascending=False)

display(cat_comparison.round(4))

joblib.dump(
    cat_filtered,
    ARTIFACT_DIR / "cat_filtered_baseline.joblib"
)

cat_comparison.to_csv(
    RESULT_DIR / "cat_baseline_comparison.csv",
    index=False
)

,Model,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds
0,CatBoost,Engineered,0.7631,0.2546,0.2445,153.2157,0.4777
1,CatBoost,Engineered + filtered,0.7629,0.2547,0.2446,147.1921,0.4558


### Select the better feature version automatically


In [31]:
if cat_filtered_results["ROC_AUC"] > cat_engineered_results["ROC_AUC"]:
    cat_feature_folder = "engineered_filtered"
    cat_feature_label = "Engineered + filtered"
    cat_selected_baseline = cat_filtered_results
else:
    cat_feature_folder = "engineered"
    cat_feature_label = "Engineered"
    cat_selected_baseline = cat_engineered_results

print("Selected:", cat_feature_label)
print("Baseline validation ROC-AUC:", cat_selected_baseline["ROC_AUC"])

feature_folder = cat_feature_folder

original_columns = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv", nrows=0
).columns.tolist()
cat_columns = [column for column in original_columns if column.startswith("categorical__")]
numeric_columns = [column for column in original_columns if column.startswith("numeric__")]

print("Categorical features:", len(cat_columns))
display(pd.DataFrame({"Categorical_Feature": cat_columns}))

if not cat_columns:
    raise ValueError("No categorical__ columns found. Check the native preprocessing CSVs.")
assert len(cat_columns) + len(numeric_columns) == len(original_columns)

column_types = {column: str for column in cat_columns}
column_types.update({column: "float32" for column in numeric_columns})

X_train = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_train.csv",
    dtype=column_types, keep_default_na=False
)
X_valid = pd.read_csv(
    PROCESSED_DIR / feature_folder / "X_valid.csv",
    dtype=column_types, keep_default_na=False
)

assert len(X_train) == len(y_train)
assert len(X_valid) == len(y_valid)
assert X_train.columns.equals(X_valid.columns)
assert np.isfinite(X_train[numeric_columns].to_numpy()).all()
assert np.isfinite(X_valid[numeric_columns].to_numpy()).all()

category_levels = {}
for column in cat_columns:
    X_train[column] = X_train[column].replace("", "Missing")
    X_valid[column] = X_valid[column].replace("", "Missing")
    levels = sorted(X_train[column].unique().tolist())
    if "Missing" not in levels:
        levels.append("Missing")
    category_levels[column] = levels
    X_valid[column] = X_valid[column].where(X_valid[column].isin(levels), "Missing")

    X_train[column] = X_train[column].astype(str)
    X_valid[column] = X_valid[column].astype(str)

input_info = {
    "original_columns": original_columns,
    "categorical_columns": cat_columns.copy(),
    "category_levels": category_levels,
    "model_columns": original_columns.copy(),
    "unknown_category_policy": "Map unseen categories to Missing"
}

joblib.dump(input_info, ARTIFACT_DIR / f"cat_{feature_folder}_input_info.joblib")
print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

pd.DataFrame([{
    "Model": "cat", "Feature_Folder": cat_feature_folder,
    "Input_Info_File": f"cat_{cat_feature_folder}_input_info.joblib",
    "Baseline_Model_File": "cat_filtered_baseline.joblib" if cat_feature_folder == "engineered_filtered" else "cat_engineered_baseline.joblib",
    "Tuned_Model_File": f"cat_{cat_feature_folder}_tuned.joblib"
}]).to_csv(RESULT_DIR / "cat_selected_configuration.csv", index=False)


Selected: Engineered
Baseline validation ROC-AUC: 0.7631235394573266
Categorical features: 16


,Categorical_Feature
0,categorical__NAME_CONTRACT_TYPE
1,categorical__CODE_GENDER
2,categorical__FLAG_OWN_CAR
3,categorical__FLAG_OWN_REALTY
4,categorical__NAME_TYPE_SUITE
5,categorical__NAME_INCOME_TYPE
6,categorical__NAME_EDUCATION_TYPE
7,categorical__NAME_FAMILY_STATUS
8,categorical__NAME_HOUSING_TYPE
9,categorical__OCCUPATION_TYPE


Training: (215256, 190)
Validation: (46127, 190)


### Optuna objective and 30-minute study



In [32]:
def cat_objective(trial):
    model = CatBoostClassifier(
        iterations=trial.suggest_int(
            "iterations", 200, 1000, step=100
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        depth=trial.suggest_int("depth", 4, 8),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg", 1, 30, log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength", 0.01, 10, log=True
        ),
        bootstrap_type="Bayesian",
        bagging_temperature=trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        thread_count=4,
        verbose=False,
        allow_writing_files=False
    )

    start = time.perf_counter()

    model.fit(X_train, y_train, cat_features=cat_columns)
    probability = model.predict_proba(X_valid)[:, 1]

    trial.set_user_attr(
        "fit_predict_seconds",
        time.perf_counter() - start
    )

    return roc_auc_score(y_valid, probability)

cat_study = optuna.create_study(
    study_name=f"cat_native_{cat_feature_folder}_matched_v1",
    storage="sqlite:///" + database_path.as_posix(),
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    load_if_exists=True
)

tuning_start = time.perf_counter()

cat_study.optimize(
    cat_objective,
    timeout=30 * 60,
    n_jobs=1,
    gc_after_trial=True
)

cat_tuning_seconds = time.perf_counter() - tuning_start

cat_study.trials_dataframe().to_csv(
    RESULT_DIR / "cat_optuna_trials.csv",
    index=False
)

print("Best validation ROC-AUC:", cat_study.best_value)
print("Best parameters:", cat_study.best_params)
print("Tuning seconds:", round(cat_tuning_seconds, 2))

[I 2026-10-16 04:36:35,857] A new study created in RDB with name: cat_native_engineered_matched_v1
[I 2026-10-16 04:42:07,157] Trial 0 finished with value: 0.761988631005249 and parameters: {'iterations': 500, 'learning_rate': 0.13125830316209655, 'depth': 7, 'l2_leaf_reg': 7.6611007077713635, 'random_strength': 0.02938027938703535, 'bagging_temperature': 0.7799726016810132}. Best is trial 0 with value: 0.761988631005249.
[I 2026-10-16 04:44:18,947] Trial 1 finished with value: 0.7516782601539069 and parameters: {'iterations': 200, 'learning_rate': 0.10440040750544663, 'depth': 7, 'l2_leaf_reg': 11.114989443094977, 'random_strength': 0.011527987128232402, 'bagging_temperature': 4.8495492608099715}. Best is trial 0 with value: 0.761988631005249.
[I 2026-10-16 04:49:42,294] Trial 2 finished with value: 0.7542900149967603 and parameters: {'iterations': 900, 'learning_rate': 0.01777174904859463, 'depth': 4, 'l2_leaf_reg': 1.8659959624904914, 'random_strength': 0.08179499475211674, 'bagging

Best validation ROC-AUC: 0.7646121141389155
Best parameters: {'iterations': 600, 'learning_rate': 0.0838375512850209, 'depth': 4, 'l2_leaf_reg': 5.748924681991978, 'random_strength': 0.59874749104614, 'bagging_temperature': 0.23225206359998862}
Tuning seconds: 1842.97


### Train the selected parameters, evaluate and save


In [33]:
cat_tuned = CatBoostClassifier(
    **cat_study.best_params,
    bootstrap_type="Bayesian",
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    thread_count=4,
    verbose=False,
    allow_writing_files=False
)

start_time = time.perf_counter()

cat_tuned.fit(X_train, y_train, cat_features=cat_columns)

cat_tuned_fit_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

valid_probability = cat_tuned.predict_proba(X_valid)[:, 1]

prediction_seconds = time.perf_counter() - start_time

cat_tuned_results = {
    "Model": "CatBoost",
    "Version": "Tuned",
    "Feature_Set": cat_feature_label,
    "ROC_AUC": roc_auc_score(y_valid, valid_probability),
    "Average_Precision": average_precision_score(
        y_valid, valid_probability
    ),
    "Log_Loss": log_loss(y_valid, valid_probability),
    "Fit_Seconds": cat_tuned_fit_seconds,
    "Prediction_Seconds": prediction_seconds,
    "Tuning_Seconds": cat_tuning_seconds
}

display(pd.DataFrame([cat_tuned_results]).round(4))

joblib.dump(
    cat_tuned,
    ARTIFACT_DIR / f"cat_{cat_feature_folder}_tuned.joblib"
)

pd.DataFrame([cat_tuned_results]).to_csv(
    RESULT_DIR / "cat_tuned_validation.csv",
    index=False
)

pd.Series(cat_study.best_params, name="Value").to_csv(
    RESULT_DIR / "cat_best_parameters.csv",
    index_label="Parameter"
)

,Model,Version,Feature_Set,ROC_AUC,Average_Precision,Log_Loss,Fit_Seconds,Prediction_Seconds,Tuning_Seconds
0,CatBoost,Tuned,Engineered,0.7646,0.2553,0.244,214.2972,0.4309,1842.9654


## Finished
